In [1]:

SID4  = 2733
SEED  = SID4
SLICE = SID4 % 1000
HP_ID = SID4 % 6
CLS_A = SID4 % 10
CLS_B = (CLS_A + 1 + ((SID4 // 10) % 9)) % 10
print(f"SID4={SID4}  SEED={SEED}  SLICE={SLICE}  HP_ID={HP_ID}  CLS_A={CLS_A}  CLS_B={CLS_B}")

SID4=2733  SEED=2733  SLICE=733  HP_ID=3  CLS_A=3  CLS_B=7


In [2]:
!pip install -q -U transformers accelerate datasets optimum gptqmodel
!rm -rf /usr/local/lib/python3.13/dist-packages/torchaudio* /usr/local/lib/python3.13/dist-packages/torchvision*

In [3]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"   # use only GPU 0 (before importing torch)

import random
import numpy as np
import torch

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

print("GPUs visible:", torch.cuda.device_count())
print("GPU name:", torch.cuda.get_device_name(0))

GPUs visible: 1
GPU name: Tesla T4


In [4]:
from importlib.metadata import version
import transformers, optimum, gptqmodel, datasets
for pkg in ["torch", "transformers", "optimum", "gptqmodel", "datasets", "accelerate"]:
    print(f"{pkg:12s} {version(pkg)}")
print("CUDA ok:", torch.cuda.is_available())


INFO  LogBar: headless/CI mode; progress animations disabled.                   
WARN  Python GIL is enabled: Multi-gpu quant acceleration for MoE models is sub-optimal and multi-core accelerated cpu packing is also disabled. We recommend Python >= 3.13.3t with Pytorch > 2.8 for mult-gpu quantization and multi-cpu packing with env `PYTHON_GIL=0`.
INFO  ENV: Auto setting PYTORCH_ALLOC_CONF='expandable_segments:True' for memory saving.


W1119 22:23:38.323000 1109 torch/utils/_pytree.py:646] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1119 22:23:38.362000 1109 torch/utils/_pytree.py:646] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


INFO  ENV: Auto setting CUDA_DEVICE_ORDER=PCI_BUS_ID for correctness.           
INFO    ┌─────────────┐    ┌────────────────────────┐    ┌────────────┐    ┌─────────┐ │ GPT-QModel  │ -> │ ▓▓▓▓▓▓▓▓▓▓▓▓ 16bit     │ -> │ ▒▒▒▒ 8bit  │ -> │ ░░ 4bit │ └─────────────┘    └────────────────────────┘    └────────────┘    └─────────┘ GPT-QModel   : 7.12.0 Transformers : 5.23.0 Torch        : 2.15.0+cu132 Triton       : 3.9.0


INFO:numexpr.utils:NumExpr defaulting to 4 threads.
INFO:datasets:TensorFlow version 2.21.0 available.
INFO:datasets:JAX version 0.11.1 available.


torch        2.15.0
transformers 5.23.0
optimum      2.3.0
gptqmodel    7.12.0
datasets     5.2.0
accelerate   1.16.0
CUDA ok: True


In [5]:
import logging, sys

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(message)s",
    handlers=[logging.FileHandler("/kaggle/working/RUN_LOG.txt", mode="w"),
              logging.StreamHandler(sys.stdout)],
    force=True,
)
log = logging.getLogger("hw11")

log.info(f"SID4={SID4} SEED={SEED} SLICE={SLICE} HP_ID={HP_ID} CLS_A={CLS_A} CLS_B={CLS_B}")
log.info(f"GPU: {torch.cuda.get_device_name(0)}")
for pkg in ["torch", "transformers", "optimum", "gptqmodel", "datasets", "accelerate"]:
    log.info(f"{pkg} {version(pkg)}")

2026-11-19 22:23:40,238 | SID4=2733 SEED=2733 SLICE=733 HP_ID=3 CLS_A=3 CLS_B=7
2026-11-19 22:23:40,240 | GPU: Tesla T4
2026-11-19 22:23:40,243 | torch 2.15.0
2026-11-19 22:23:40,246 | transformers 5.23.0
2026-11-19 22:23:40,248 | optimum 2.3.0
2026-11-19 22:23:40,251 | gptqmodel 7.12.0
2026-11-19 22:23:40,254 | datasets 5.2.0
2026-11-19 22:23:40,256 | accelerate 1.16.0


In [6]:
from huggingface_hub import HfApi

MODEL_ID = "facebook/opt-125m"  
REVISION = HfApi().model_info(MODEL_ID).sha

log.info(f"MODEL_ID={MODEL_ID}")
log.info(f"REVISION={REVISION}")

2026-11-19 22:23:40,415 | HTTP Request: GET https://huggingface.co/api/models/facebook/opt-125m "HTTP/1.1 200 OK"
2026-11-19 22:23:40,418 | MODEL_ID=facebook/opt-125m
2026-11-19 22:23:40,419 | REVISION=27dcfa74d334bc871f3234de431e71c6eeba5dd6


In [7]:
from transformers import AutoTokenizer
from datasets import load_dataset

logging.getLogger("httpx").setLevel(logging.WARNING)   # hide the HTTP noise

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=REVISION)

test_ds = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split="test")
test_text = "\n\n".join(test_ds["text"])
test_enc = tokenizer(test_text, return_tensors="pt")

log.info(f"Wikitext2 test tokens: {test_enc.input_ids.shape[1]}")

2026-11-19 22:23:40,761 | Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-11-19 22:23:43,150 | Wikitext2 test tokens: 287645


In [8]:
import math
from tqdm.auto import tqdm

@torch.no_grad()
def wikitext_ppl(model, enc=test_enc, max_length=2048, stride=512):
    device = next(model.parameters()).device
    seq_len = enc.input_ids.size(1)
    nll_sum, n_tokens, prev_end = 0.0, 0, 0

    for begin in tqdm(range(0, seq_len, stride), desc="ppl"):
        end = min(begin + max_length, seq_len)
        trg_len = end - prev_end                 
        input_ids = enc.input_ids[:, begin:end].to(device)
        labels = input_ids.clone()
        labels[:, :-trg_len] = -100              

        loss = model(input_ids, labels=labels).loss
        n = (labels[:, 1:] != -100).sum().item() 
        nll_sum += loss.float().item() * n
        n_tokens += n

        prev_end = end
        if end == seq_len:
            break

    avg_nll = nll_sum / n_tokens
    return math.exp(avg_nll), avg_nll

In [9]:
import time


PROMPT_IDS = test_enc.input_ids[:, SLICE:SLICE + 128]
NEW_TOKENS = 64
WARMUP, RUNS = 5, 30

@torch.no_grad()
def measure_latency(model):
    device = next(model.parameters()).device
    ids = PROMPT_IDS.to(device)
    attn = torch.ones_like(ids)
    gen_kwargs = dict(attention_mask=attn, max_new_tokens=NEW_TOKENS,
                      min_new_tokens=NEW_TOKENS, do_sample=False,
                      pad_token_id=tokenizer.pad_token_id)

    for _ in range(WARMUP):                     
        model.generate(ids, **gen_kwargs)

    torch.cuda.reset_peak_memory_stats()
    times = []
    for _ in range(RUNS):
        torch.cuda.synchronize()
        t0 = time.perf_counter()
        model.generate(ids, **gen_kwargs)
        torch.cuda.synchronize()
        times.append((time.perf_counter() - t0) * 1000)   # ms

    peak_mb = torch.cuda.max_memory_allocated() / 1024**2
    p50, p95 = np.percentile(times, [50, 95])
    return p50, p95, peak_mb

In [10]:
from pathlib import Path

def dir_size_mb(path):
    return sum(f.stat().st_size for f in Path(path).rglob("*") if f.is_file()) / 1024**2

results = {}  

In [11]:
import gc
from transformers import AutoModelForCausalLM

BASE_DIR = "/kaggle/working/base_fp16"

log.info("=== BASE fp16 ===")
base = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, revision=REVISION, dtype=torch.float16, device_map={"": 0}
).eval()


torch.cuda.reset_peak_memory_stats()
ppl, nll = wikitext_ppl(base)
eval_peak_mb = torch.cuda.max_memory_allocated() / 1024**2
log.info(f"BASE ppl={ppl:.3f} eval_loss={nll:.4f} eval_peak_mem={eval_peak_mb:.1f}MB")


p50, p95, inf_peak_mb = measure_latency(base)
log.info(f"BASE p50={p50:.1f}ms p95={p95:.1f}ms infer_peak_mem={inf_peak_mb:.1f}MB")


base.save_pretrained(BASE_DIR)
disk_mb = dir_size_mb(BASE_DIR)
log.info(f"BASE disk={disk_mb:.1f}MB")

results["base_fp16"] = dict(ppl=ppl, eval_loss=nll, p50_ms=p50, p95_ms=p95,
                            infer_peak_mb=inf_peak_mb, eval_peak_mb=eval_peak_mb,
                            disk_mb=disk_mb)


del base; gc.collect(); torch.cuda.empty_cache()

2026-11-19 22:23:43,212 | === BASE fp16 ===


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

ppl:   0%|          | 0/562 [00:00<?, ?it/s]

2026-11-19 22:24:08,343 | BASE ppl=23.687 eval_loss=3.1649 eval_peak_mem=1295.8MB
2026-11-19 22:24:27,789 | BASE p50=548.4ms p95=576.0ms infer_peak_mem=254.2MB


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

2026-11-19 22:24:28,231 | BASE disk=238.9MB


In [12]:
from transformers import GPTQConfig

QUANT_DIR = "/kaggle/working/opt125m_gptq_4bit"

gptq_config = GPTQConfig(
    bits=4,
    group_size=128,
    dataset="wikitext2",     
    tokenizer=tokenizer,
    desc_act=False,
)

log.info("=== GPTQ quantization (4-bit, group_size=128, calib=wikitext2 train) ===")
t0 = time.perf_counter()
qmodel = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, revision=REVISION, dtype=torch.float16,
    device_map={"": 0}, quantization_config=gptq_config,
)
quant_time = time.perf_counter() - t0
log.info(f"GPTQ quantization time: {quant_time:.1f}s")

qmodel.save_pretrained(QUANT_DIR)
tokenizer.save_pretrained(QUANT_DIR)
log.info(f"Saved quantized model to {QUANT_DIR}")

del qmodel; gc.collect(); torch.cuda.empty_cache()

2026-11-19 22:24:28,609 | === GPTQ quantization (4-bit, group_size=128, calib=wikitext2 train) ===


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

Quantizing model.decoder.layers blocks :   0%|          | 0/12 [00:00<?, ?it/s]

2026-11-19 22:24:31,394 | Start quantizing block model.decoder.layers 1/12
2026-11-19 22:24:31,395 | Module to quantize [['self_attn.k_proj'], ['self_attn.v_proj'], ['self_attn.q_proj'], ['self_attn.out_proj'], ['fc1'], ['fc2']]


Quantizing layers inside the block:   0%|          | 0/6 [00:00<?, ?it/s]

2026-11-19 22:24:31,706 | Quantizing self_attn.k_proj in block 1/12...
2026-11-19 22:24:32,256 | Quantizing self_attn.v_proj in block 1/12...
2026-11-19 22:24:32,636 | Quantizing self_attn.q_proj in block 1/12...
2026-11-19 22:24:33,014 | Quantizing self_attn.out_proj in block 1/12...
2026-11-19 22:24:33,392 | Quantizing fc1 in block 1/12...
2026-11-19 22:24:34,325 | Quantizing fc2 in block 1/12...
2026-11-19 22:24:35,255 | Start quantizing block model.decoder.layers 2/12
2026-11-19 22:24:35,256 | Module to quantize [['self_attn.k_proj'], ['self_attn.v_proj'], ['self_attn.q_proj'], ['self_attn.out_proj'], ['fc1'], ['fc2']]


... (22 outputs left out)

/usr/local/lib/python3.13/dist-packages/transformers/modeling_utils.py:3303: FutureWarning: `is_gradient_checkpointing` is deprecated and will be removed in v6. Use `is_activation_checkpointing` instead.
  warnings.warn(
[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


2026-11-19 22:25:15,539 | model.decoder.layers.0.self_attn.k_proj
2026-11-19 22:25:15,546 | model.decoder.layers.0.self_attn.out_proj
2026-11-19 22:25:15,551 | model.decoder.layers.0.self_attn.q_proj
2026-11-19 22:25:15,558 | model.decoder.layers.0.self_attn.v_proj
2026-11-19 22:25:15,563 | model.decoder.layers.0.fc1
2026-11-19 22:25:15,579 | model.decoder.layers.0.fc2
2026-11-19 22:25:15,595 | model.decoder.layers.1.self_attn.k_proj
2026-11-19 22:25:15,599 | model.decoder.layers.1.self_attn.out_proj
2026-11-19 22:25:15,603 | model.decoder.layers.1.self_attn.q_proj
2026-11-19 22:25:15,607 | model.decoder.layers.1.self_attn.v_proj
2026-11-19 22:25:15,611 | model.decoder.layers.1.fc1
2026-11-19 22:25:15,628 | model.decoder.layers.1.fc2
2026-11-19 22:25:15,644 | model.decoder.layers.2.self_attn.k_proj
2026-11-19 22:25:15,648 | model.decoder.layers.2.self_attn.out_proj
2026-11-19 22:25:15,652 | model.decoder.layers.2.self_attn.q_proj
2026-11-19 22:25:15,656 | model.decoder.layers.2.self_at

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

2026-11-19 22:25:17,897 | Saved quantized model to /kaggle/working/opt125m_gptq_4bit


In [13]:
from gptqmodel import GPTQModel, BACKEND

log.info("=== GPTQ 4-bit (reloaded from disk, backend=ExLlamaV2; Marlin needs sm_80+, T4 is sm_75) ===")
qwrap = GPTQModel.load(QUANT_DIR, backend=BACKEND.EXLLAMA_V2, device="cuda:0")
qmodel = qwrap.model.eval()

# quality
torch.cuda.reset_peak_memory_stats()
ppl, nll = wikitext_ppl(qmodel)
eval_peak_mb = torch.cuda.max_memory_allocated() / 1024**2
log.info(f"GPTQ ppl={ppl:.3f} eval_loss={nll:.4f} eval_peak_mem={eval_peak_mb:.1f}MB")

# speed + inference memory
p50, p95, inf_peak_mb = measure_latency(qmodel)
log.info(f"GPTQ p50={p50:.1f}ms p95={p95:.1f}ms infer_peak_mem={inf_peak_mb:.1f}MB")

# disk size
disk_mb = dir_size_mb(QUANT_DIR)
log.info(f"GPTQ disk={disk_mb:.1f}MB")

results["gptq_4bit"] = dict(ppl=ppl, eval_loss=nll, p50_ms=p50, p95_ms=p95,
                            infer_peak_mb=inf_peak_mb, eval_peak_mb=eval_peak_mb,
                            disk_mb=disk_mb, quant_time_s=quant_time)

del qmodel, qwrap; gc.collect(); torch.cuda.empty_cache()

2026-11-19 22:25:18,289 | === GPTQ 4-bit (reloaded from disk, backend=ExLlamaV2; Marlin needs sm_80+, T4 is sm_75) ===
from_quantized: adapter: None
INFO  Loader: Auto dtype (native float16): `torch.float16`                      
INFO  QuantizeConfig: Ignoring unknown parameter in the quantization configuration: backend.
INFO  QuantizeConfig: Ignoring unknown parameter in the quantization configuration: batch_size.
INFO  QuantizeConfig: Ignoring unknown parameter in the quantization configuration: block_name_to_quantize.
INFO  QuantizeConfig: Ignoring unknown parameter in the quantization configuration: cache_block_outputs.
INFO  QuantizeConfig: Ignoring unknown parameter in the quantization configuration: dataset.
INFO  QuantizeConfig: Ignoring unknown parameter in the quantization configuration: max_input_length.
INFO  QuantizeConfig: Ignoring unknown parameter in the quantization configuration: model_seqlen.
INFO  QuantizeConfig: Ignoring unknown parameter in the quantization config

ppl:   0%|          | 0/562 [00:00<?, ?it/s]

2026-11-19 22:26:33,175 | GPTQ ppl=28.414 eval_loss=3.3469 eval_peak_mem=1176.6MB
2026-11-19 22:27:08,665 | GPTQ p50=1012.6ms p95=1035.0ms infer_peak_mem=150.1MB
2026-11-19 22:27:08,667 | GPTQ disk=122.7MB


In [14]:
import json, pandas as pd

b, g = results["base_fp16"], results["gptq_4bit"]
table = pd.DataFrame({
    "Configuration": ["Base LLM (fp16)", "GPTQ-quantized LLM (4-bit, g128)"],
    "Wikitext2 PPL": [f"{b['ppl']:.2f}", f"{g['ppl']:.2f}"],
    "p50 / p95 latency (ms)": [f"{b['p50_ms']:.1f} / {b['p95_ms']:.1f}", f"{g['p50_ms']:.1f} / {g['p95_ms']:.1f}"],
    "Peak memory (GB)": [f"{b['infer_peak_mb']/1024:.3f}", f"{g['infer_peak_mb']/1024:.3f}"],
    "Size on disk (MB)": [f"{b['disk_mb']:.1f}", f"{g['disk_mb']:.1f}"],
})
print("Table HW11.1 - GPTQ comparison on the same LLM/task")
print(table.to_string(index=False))

with open("/kaggle/working/results_part1.json", "w") as f:
    json.dump(results, f, indent=2)
log.info("Saved results_part1.json")

Table HW11.1 - GPTQ comparison on the same LLM/task
                   Configuration Wikitext2 PPL p50 / p95 latency (ms) Peak memory (GB) Size on disk (MB)
                 Base LLM (fp16)         23.69          548.4 / 576.0            0.248             238.9
GPTQ-quantized LLM (4-bit, g128)         28.41        1012.6 / 1035.0            0.147             122.7
2026-11-19 22:27:09,104 | Saved results_part1.json
